In [ ]:
import numpy as np
import psutil
import subprocess
import random as rd
import pickle
import gc
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import LogarithmicMapper

def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")


def single_line (lines):
    return ''.join(lines.splitlines())

In [2]:
n_x = 12
n_y = 1
n_site = n_x * n_y
n_qubit = n_site
dim = 2**n_qubit

n_qubit_circuit = n_qubit + 1

n_dimer = n_site//2

core  = 0
cores = 1

In [3]:
seed_transpiler = 42

In [4]:
from qiskit import QuantumRegister, QuantumCircuit
qr_circuit = QuantumRegister(n_qubit_circuit,name='q')
#qr = qr_circuit[1:]
#qm = qr_circuit[0]
#indx_qm = 0
indx_qm = n_qubit//2
qm = qr_circuit[indx_qm]
qr = qr_circuit[:indx_qm] + qr_circuit[indx_qm+1:]
#print(qm,qr)

In [5]:
J                   = 1.0
Delta               = -1.0
h                   = 0.0

In [6]:
t_inters = [0.0, 1.0]
#t_inters            = [0.0, 1.0]
n_inter             = len(t_inters)
hamiltonians        = []
mapper = LogarithmicMapper()
for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    hamiltonian_list = []
    # intra-dimer terms
    for i in range(0,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*Delta/4)
        hamiltonian_list.append(term)
        term = ('Z',[ii],-h/2)
        hamiltonian_list.append(term)
        term = ('Z',[jj],-h/2)
        hamiltonian_list.append(term)
    # inter-dimer terms
    for i in range(1,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*t_inter*Delta/4)
        hamiltonian_list.append(term)
    print(hamiltonian_list)
    hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonians.append(hamiltonian.simplify())

    if (core==0):
        print(t_inter, single_line(str(hamiltonians[i_inter])))
        print('')

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [4, 5], -0.25), ('YY', [4, 5], -0.25), ('ZZ', [4, 5], 0.25), ('Z', [4], -0.0), ('Z', [5], -0.0), ('XX', [6, 7], -0.25), ('YY', [6, 7], -0.25), ('ZZ', [6, 7], 0.25), ('Z', [6], -0.0), ('Z', [7], -0.0), ('XX', [8, 9], -0.25), ('YY', [8, 9], -0.25), ('ZZ', [8, 9], 0.25), ('Z', [8], -0.0), ('Z', [9], -0.0), ('XX', [10, 11], -0.25), ('YY', [10, 11], -0.25), ('ZZ', [10, 11], 0.25), ('Z', [10], -0.0), ('Z', [11], -0.0), ('XX', [1, 2], -0.0), ('YY', [1, 2], -0.0), ('ZZ', [1, 2], 0.0), ('XX', [3, 4], -0.0), ('YY', [3, 4], -0.0), ('ZZ', [3, 4], 0.0), ('XX', [5, 6], -0.0), ('YY', [5, 6], -0.0), ('ZZ', [5, 6], 0.0), ('XX', [7, 8], -0.0), ('YY', [7, 8], -0.0), ('ZZ', [7, 8], 0.0), ('XX', [9, 10], -0.0), ('YY', [9, 10], -0.0), ('ZZ', [9, 10], 0.0)]
0.0 SparsePauliOp(['IIIIIIIIIIXX', 'III

In [7]:
init_circuit = QuantumCircuit(qr_circuit)
# init circuit for staring from dimers
n_dimer_half = n_dimer//2
index_attached_to_qm = n_qubit//2-1

#  geometry ; qm -- qr[n_qubit//2-1] -- ... -- qr[1] -- qr[0]
#                   \--qr[n_qubit//2] -- ... -- qr[n_qubit-1]
# apply cxs first
init_circuit.cx(qm,qr[index_attached_to_qm])
for i_qubit in range(index_attached_to_qm,1,-1):
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
for i_qubit in range(index_attached_to_qm,n_qubit-2):
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])


# apply chs + z + cx
init_circuit.ch(qr[1],qr[0])
init_circuit.cx(qr[0],qr[1])
for i_dimer in range(1,n_dimer-1):
    i_qubit = 2*i_dimer
    init_circuit.ch(qr[i_qubit+1],qr[i_qubit])
    init_circuit.z(qr[i_qubit])
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])

init_circuit.ch(qr[-2],qr[-1])
init_circuit.cx(qr[-1],qr[-2])

#
#for i_dimer in range(1,n_dimer_half+1):
#    i_qubit = 2*(i_dimer+n_dimer_half)
#    init_circuit.s(qr[i_qubit+1])
#    init_circuit.h(qr[i_qubit+1])
#    init_circuit.t(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])
#    init_circuit.tdg(qr[i_qubit+1])
#    if (i_dimer<n_dimer_half):
#        init_circuit.sx(qr[i_qubit+1])
#        init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])
#        init_circuit.x(qr[i_qubit+1])
#        init_circuit.h(qr[i_qubit+1])
#    else:
#        init_circuit.h(qr[i_qubit+1])
#        init_circuit.sdg(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit+1],qr[i_qubit])




print(init_circuit)
init_circuit_inv = init_circuit.inverse()
print(init_circuit_inv)

                               ┌───┐                    
 q_0: ─────────────────────────┤ H ├──■─────────────────
                          ┌───┐└─┬─┘┌─┴─┐               
 q_1: ────────────────────┤ X ├──■──┤ X ├───────────────
                     ┌───┐└─┬─┘┌───┐├───┤               
 q_2: ───────────────┤ X ├──■──┤ H ├┤ Z ├──■────────────
                ┌───┐└─┬─┘     └─┬─┘└───┘┌─┴─┐          
 q_3: ──────────┤ X ├──■─────────■───────┤ X ├──────────
           ┌───┐└─┬─┘┌───┐┌───┐          └───┘          
 q_4: ─────┤ X ├──■──┤ H ├┤ Z ├──■──────────────────────
      ┌───┐└─┬─┘     └─┬─┘└───┘┌─┴─┐                    
 q_5: ┤ X ├──■────■────■───────┤ X ├────────────────────
      └─┬─┘       │            └───┘                    
 q_6: ──■─────────┼─────────────────────────────────────
                ┌─┴─┐          ┌───┐┌───┐               
 q_7: ──────────┤ X ├──■───────┤ H ├┤ Z ├──■────────────
                └───┘┌─┴─┐     └─┬─┘└───┘┌─┴─┐          
 q_8: ───────────────┤ X ├──■──

In [8]:
n_hamiltonians = len(hamiltonians)
if (core==0):
    print('# Hamiltonian differences')
hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs[alpha])))

if (core==0):
    print('# Hamiltonian differences_list')
hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_list[alpha])

if (core==0):
    print('# Reduced Hamiltonian differences_list')
hamiltonian_diffs_reduced = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_list = []
    if (n_site>=4) and (n_dimer%2==0):
        # for 4 site
        # pos # 1
        factor_XX = 2  # 1 for XX, 1 for YY
        factor_ZZ = -1 # -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm
        jj = index_attached_to_qm+1
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=6) and (n_dimer%2==1):
        # for 6 site
        # pos # 1
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm
        jj = index_attached_to_qm-1
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=8) and (n_dimer%2==0):
        # for 8 site
        # pos # 2
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm-1
        jj = index_attached_to_qm-2
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=10) and (n_dimer%2==1):
        # for 10 site
        # pos # 2
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm-2
        jj = index_attached_to_qm-3
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=12) and (n_dimer%2==0):
        # for 12 site
        # pos # 3
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm-3
        jj = index_attached_to_qm-4
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)

    print(hamiltonian_list)
    dH = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonian_diffs_reduced.append(dH.simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs_reduced[alpha])))
if (core==0):
    print('# Hamiltonian differences_reduced_list')
hamiltonian_diffs_reduced_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_reduced_list.append(hamiltonian_diffs_reduced[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_reduced_list[alpha])

# Hamiltonian differences
0 SparsePauliOp(['IIIIIIIIIXXI', 'IIIIIIIIIYYI', 'IIIIIIIIIZZI', 'IIIIIIIXXIII', 'IIIIIIIYYIII', 'IIIIIIIZZIII', 'IIIIIXXIIIII', 'IIIIIYYIIIII', 'IIIIIZZIIIII', 'IIIXXIIIIIII', 'IIIYYIIIIIII', 'IIIZZIIIIIII', 'IXXIIIIIIIII', 'IYYIIIIIIIII', 'IZZIIIIIIIII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])
# Hamiltonian differences_list
[('IIIIIIIIIXXI', (-0.25+0j)), ('IIIIIIIIIYYI', (-0.25+0j)), ('IIIIIIIIIZZI', (0.25+0j)), ('IIIIIIIXXIII', (-0.25+0j)), ('IIIIIIIYYIII', (-0.25+0j)), ('IIIIIIIZZIII', (0.25+0j)), ('IIIIIXXIIIII', (-0.25+0j)), ('IIIIIYYIIIII', (-0.25+0j)), ('IIIIIZZIIIII', (0.25+0j)), ('IIIXXIIIIIII', (-0.25+0j)), ('IIIYYIIIIIII', (-0.25+0j)), ('IIIZZIIIIIII', (0.25+0j)), ('IXXIIIIIIIII', (-0.25+0j)), ('IYYIIIIIIIII', (-0.25+0j)), ('IZZIIIIIIIII', (0.25+0j))]
# Reduced Hamiltonian differences_list
[('XX', [5, 6

In [9]:
nmc = 300
beta = 2.0
n_shot = 2048

n_obs = 3
# 0; norm, 1; dE1, 2; dE2
O_timelists         = [[[[] for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]

#exact_dir = '../exact'
#with open(exact_dir+'/XXZ10.time.binary','rb') as file_:
#    [O_timelists] = pickle.load(file_)

exact_dir = '../../'+str(n_site)+'/exact'
with open(exact_dir+'/XXZ'+str(n_site)+'.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

In [10]:
observable = SparsePauliOp.from_sparse_list([("Z", [indx_qm], 1)], num_qubits=n_qubit_circuit)
print(observable)

SparsePauliOp(['IIIIIIZIIIIII'],
              coeffs=[1.+0.j])


In [11]:
def Apply_R_XXplusYYplusZZ(theta_x,theta_y,theta_z, jj, qc_: QuantumCircuit):
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(theta_z,qr[jj])
    qc_.rz(theta_x+np.pi/2,qr[jj+1])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(-theta_y,qr[jj])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.sx(qr[jj])
    qc_.sxdg(qr[jj+1])
    


In [12]:
def ApplyConsecutiveTrotterEvolution_from_dimer_solution(times, alphas, n_trotters, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    # only consider indx==1 case
    # intra-dimer evolution first
    # skip the first evolution
    # phase corresponding to this part should be considered properly.
    # for i_qubit in range(0,n_qubit,2):
    #     Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
    #                            thetas_yy_intra[0]/2.0,
    #                            thetas_zz_intra[0]/2.0,
    #                            i_qubit, qc_)
    for i_time in range(n_time-1):
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                   thetas_yy_inter[i_time],
                                   thetas_zz_inter[i_time],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[i_time]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                   (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                   (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                   i_qubit, qc_)
    # last evolution
    # inter-dimer evolution
    for i_qubit in range(1,n_qubit-1,2):
        Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                               thetas_yy_inter[-1],
                               thetas_zz_inter[-1],
                               i_qubit, qc_)
    for i_trotter in range(n_trotters[-1]-1):
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
    # intra-dimer evolution
    for i_qubit in range(0,n_qubit,2):
        Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
                               (thetas_yy_intra[-1])/2.0,
                               (thetas_zz_intra[-1])/2.0,
                               i_qubit, qc_)

def ApplyConsecutiveTrotterEvolution_to_dimer_solution(times, alphas, n_trotters, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    # only consider indx==1 case
    # intra-dimer evolution first
    for i_qubit in range(0,n_qubit,2):
        Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
                               thetas_yy_intra[0]/2.0,
                               thetas_zz_intra[0]/2.0,
                               i_qubit, qc_)
    for i_time in range(n_time-1):
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                   thetas_yy_inter[i_time],
                                   thetas_zz_inter[i_time],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[i_time]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                   (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                   (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                   i_qubit, qc_)
    # last evolution
    # inter-dimer evolution
    for i_qubit in range(1,n_qubit-1,2):
        Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                               thetas_yy_inter[-1],
                               thetas_zz_inter[-1],
                               i_qubit, qc_)
    for i_trotter in range(n_trotters[-1]-1):
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
    # intra-dimer evolution
    # skip the last evolution
    # phase corresponding to this part should be considered properly.
    # for i_qubit in range(0,n_qubit,2):
    #     Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
    #                            (thetas_yy_intra[-1])/2.0,
    #                            (thetas_zz_intra[-1])/2.0,
    #                            i_qubit, qc_)

In [13]:
def ApplyConsecutiveTrotterEvolution(times, alphas, n_trotters, indx, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    if (indx==0):
        # inter-dimer evolution first
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[0]/2.0,
                                   thetas_yy_inter[0]/2.0,
                                   thetas_zz_inter[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_inter[i_time]+thetas_xx_inter[i_time+1])/2.0,
                                       (thetas_yy_inter[i_time]+thetas_yy_inter[i_time+1])/2.0,
                                       (thetas_zz_inter[i_time]+thetas_zz_inter[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_inter[-1])/2.0,
                                   (thetas_yy_inter[-1])/2.0,
                                   (thetas_zz_inter[-1])/2.0,
                                   i_qubit, qc_)
    elif (indx==1):
        # intra-dimer evolution first
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
                                   thetas_yy_intra[0]/2.0,
                                   thetas_zz_intra[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                       (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                       (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
                                   (thetas_yy_intra[-1])/2.0,
                                   (thetas_zz_intra[-1])/2.0,
                                   i_qubit, qc_)

In [14]:
# read exact values
nsec = n_qubit//2 + 1
exact_dir = '../../'+str(n_site)+'/exact'
norms_exact = np.zeros((nsec,n_hamiltonians), dtype=float)
eigen_energies_exact = np.zeros((nsec,n_hamiltonians),dtype=float)
with open(exact_dir + '/exact.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        for isec in range(nsec):
            norms_exact[isec,alpha] = float(ls[isec+1])
            eigen_energies_exact[isec,alpha] = float(ls[isec+nsec+1])
        alpha += 1
for isec in range(nsec):
    for alpha in range(n_hamiltonians):
        print(isec,alpha,norms_exact[isec,alpha],eigen_energies_exact[isec,alpha])

0 0 1.0 1.5
0 1 1.0 2.75
1 0 1.0 0.5
1 1 0.0219486765639021 0.7840741737109318
2 0 1.0 -0.5000000000000004
2 1 0.011255053137286721 -1.0672053172339946
3 0 1.0 -1.5000000000000013
3 1 0.028647604452232474 -2.7038933379778776
4 0 1.0 -2.5000000000000018
4 1 0.0035227066840307466 -4.009912795647325
5 0 1.0 -3.5000000000000027
5 1 0.0109538276772967 -4.861147937036375
6 0 1.0 -4.500000000000001
6 1 0.6057656282247224 -5.142090632840531


In [15]:
def NumberOfTrotterSteps(alpha):
    return 2
if (core==0):
    print('# of Trotter Steps for each alpha')
    for alpha_ in range(1,n_hamiltonians):
        print(NumberOfTrotterSteps(alpha_))

# of Trotter Steps for each alpha
2


In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
 
# Load saved credentials
service = QiskitRuntimeService()
backend_name = "ibm_torino" 
backend = service.backend(name=backend_name)

# use fake backend
# from qiskit_ibm_runtime.fake_provider import FakeTorino
# backend = FakeTorino()
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
options = EstimatorOptions()
options.default_shots = n_shot
options.resilience_level = 1
options.dynamical_decoupling.enable = True
#options.twirling.enable_gates = True
options.dynamical_decoupling.sequence_type = "XY4"
estimator = Estimator(backend, options=options)
max_circuits = backend.max_circuits


In [17]:
#initial_layout = [100,101,111,102,103] # seems to best at 241111, 3:52 PM, no
# initial_layout = [96,97,110,98,99] # seems to best at 241106, 4:46 PM
if (n_site==4):
    initial_layout = [30, 31, 18, 32, 33] # seems to best at 241115, 8:16 AM
if (n_site==6):
    initial_layout = [29, 30, 31, 18, 32, 33, 37] # seems to best at 241115, 8:16 AM
if (n_site==8):
    initial_layout = [28, 29, 30, 31, 18, 32, 33, 37, 52] # seems to best at 241115, 8:16 AM
if (n_site==10):
    initial_layout = [27, 28, 29, 30, 31, 18, 32, 33, 37, 52, 51] # seems to best at 241115, 8:16 AM
if (n_site==12):
    initial_layout = [26, 27, 28, 29, 30, 31, 18, 32, 33, 37, 52, 51, 50] # seems to best at 241115, 8:16 AM

In [18]:
print(initial_layout)

[26, 27, 28, 29, 30, 31, 18, 32, 33, 37, 52, 51, 50]


In [19]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm3 = generate_preset_pass_manager(initial_layout=initial_layout, backend=backend, optimization_level=3, seed_transpiler=seed_transpiler)
pm1 = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=seed_transpiler)

In [20]:
#from qiskit import qpy
#backend_options = {'method':'statevector', 'max_parallel_threads':1}
#if (core==0):
#    code_run_pass_manager = '''from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
#from qiskit import qpy
#from qiskit_aer import AerSimulator
#backend_options = '''+ str(backend_options) +'''
#sim = AerSimulator(**backend_options)
#pass_manager = generate_preset_pass_manager(2, sim)
#with open('circuits.qpy', 'rb') as file_:
#    circuits = qpy.load(file_)
#    n_circuit = len(circuits)
#    circuits_opt = []
#    for i_circuit in range(n_circuit):
#        circuit_opt = pass_manager.run(circuits[i_circuit])
#        circuits_opt.append(circuit_opt)
#with open ('circuits_opt.qpy', 'wb') as file_:
#    qpy.dump(circuits_opt,file_)
#    '''
#    
#    with open('run_pass_manager.py', 'w') as file_:
#        file_.write(code_run_pass_manager)

In [21]:
# make a circuit
nhd = len(hamiltonian_diffs_reduced_list[0]) # same difference elements for all
circuits = []
# for n_qubit>=4, even dimer
# CXX, at pos # 1
if (n_site>=4) and (n_dimer%2==0):
    # CXX_(m,0,1)
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
    circuits.append(circuit)
# for n_qubit>=6, odd dimer
# CXX, at pos # 1
if (n_site>=6) and (n_dimer%2==1):
    # CXX_(m,0,-1)
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuits.append(circuit)
# for n_qubit>=8, even dimer
# CXX, at pos # 2
if (n_site>=8) and (n_dimer%2==0):
    # CXX_(m,-1,-2), 6 CNOTs
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuits.append(circuit)
# for n_qubit>=10, odd dimer
# CXX, at pos # 2
if (n_site>=10) and (n_dimer%2==1):
    # CXX_(m,-2,-3), 10 CNOTs
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
    # CX (m,-2)
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])

    circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])

    circuits.append(circuit)
# for n_qubit>=12, even dimer
# CXX, at pos # 3
if (n_site>=12) and (n_dimer%2==0):
    # CXX_(m,-3,-4)
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm-3],qr[index_attached_to_qm-4])
    # CX (m,-3)
    circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])

    circuit.cx(qr[index_attached_to_qm-3],qr[index_attached_to_qm-4])
    circuits.append(circuit)


# # CZZ, at pos # 1
# circuit = QuantumCircuit(qr_circuit)
# circuit.h(qr[index_attached_to_qm])
# circuit.h(qr[index_attached_to_qm+1])
# circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
# circuit.cx(qm,qr[index_attached_to_qm])
# circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
# circuit.h(qr[index_attached_to_qm+1])
# circuit.h(qr[index_attached_to_qm])
# circuits.append(circuit)

# CZZ, at pos # 1
#circuit = QuantumCircuit(qr_circuit)
#circuit.h(qr[index_attached_to_qm])
#circuit.h(qr[index_attached_to_qm-1])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.h(qr[index_attached_to_qm-1])
#circuit.h(qr[index_attached_to_qm])
#circuits.append(circuit)

# for n_qubit>=8, odd dimer
# # CXX, at pos # 2
#circuit = QuantumCircuit(qr_circuit)
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuits.append(circuit)
## CZZ, at pos # 2
#circuit = QuantumCircuit(qr_circuit)
#circuit.h(qr[index_attached_to_qm-2])
#circuit.h(qr[index_attached_to_qm-3])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.h(qr[index_attached_to_qm-2])
#circuit.h(qr[index_attached_to_qm-3])
#circuits.append(circuit)

In [22]:
for circuit in circuits:
    print(circuit)

                     
 q_0: ───────────────
                     
 q_1: ───────────────
                     
 q_2: ───────────────
                     
 q_3: ───────────────
                     
 q_4: ───────────────
           ┌───┐     
 q_5: ──■──┤ X ├──■──
        │  └─┬─┘  │  
 q_6: ──┼────■────┼──
      ┌─┴─┐     ┌─┴─┐
 q_7: ┤ X ├─────┤ X ├
      └───┘     └───┘
 q_8: ───────────────
                     
 q_9: ───────────────
                     
q_10: ───────────────
                     
q_11: ───────────────
                     
q_12: ───────────────
                     
                               
 q_0: ─────────────────────────
                               
 q_1: ─────────────────────────
                               
 q_2: ─────────────────────────
      ┌───┐               ┌───┐
 q_3: ┤ X ├───────────────┤ X ├
      └─┬─┘┌───┐     ┌───┐└─┬─┘
 q_4: ──■──┤ X ├─────┤ X ├──■──
           └─┬─┘┌───┐└─┬─┘┌───┐
 q_5: ───────■──┤ X ├──■──┤ X ├
                └─┬─┘ 

In [23]:
#if (core==0):
#    with open ('circuits.qpy', 'wb') as file_:
#        qpy.dump(circuits,file_)
#    pass_manager_run = subprocess.run(['python3', 'run_pass_manager.py'])

In [24]:
#with open('circuits_opt.qpy', 'rb') as file_:
#    paulis_opt = qpy.load(file_)
#for circuit in paulis_opt:
#    print(circuit)

In [25]:
paulis_opt = []
for circuit in circuits:
    circuit_opt = pm3.run(circuit)
    paulis_opt.append(circuit_opt) 

In [26]:
paulis_opt[0].draw(idle_wires=False)

»
q_6 -> 18 ────────────────────────────────────────────────────────■───────»
                                       ┌─────────┐┌────┐┌───────┐ │ ┌────┐»
q_5 -> 31 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─┤ √X ├»
          ┌─────────┐┌────┐┌───────┐ │ └─────────┘└────┘└───────┘   └────┘»
q_7 -> 32 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────────────────────────────»
          └─────────┘└────┘└───────┘                                      »
«                                         
«q_6 -> 18 ───────────────────────────────
«          ┌─────────┐                    
«q_5 -> 31 ┤ Rz(π/2) ├─■──────────────────
«          └─────────┘ │ ┌────┐┌─────────┐
«q_7 -> 32 ────────────■─┤ √X ├┤ Rz(π/2) ├
«                        └────┘└─────────┘

In [27]:
paulis_opt[1].draw(idle_wires=False)

»
q_6 -> 18 ─────────────────────────────────────────────────────────────────────»
          ┌─────────┐┌────┐┌───────┐                                           »
q_3 -> 29 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─────────────────────────────────────────»
          └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐              »
q_4 -> 30 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────»
                                       └─────────┘└────┘└───────┘ │ ┌─────────┐»
q_5 -> 31 ────────────────────────────────────────────────────────■─┤ Rz(π/2) ├»
                                                                    └─────────┘»
«                                                                           »
«q_6 -> 18 ────────────────■────────────────────────────────────────■───────»
«                          │                                        │ ┌────┐»
«q_3 -> 29 ────────────────┼─────────────────────────────────────■──┼─┤ √X ├»
«                          │                   ┌────┐┌─────────┐ │  │ └────┘»
«q_4 -> 30 ────────────────┼─────────────────■─┤ √X ├┤ Rz(π/2) ├─■──┼───────»
«          ┌────┐┌───────┐ │ ┌────┐┌───────┐ │ ├────┤└┬───────┬┘    │ ┌────┐»
«q_5 -> 31 ┤ √X ├┤ Rz(π) ├─■─┤ √X ├┤ Rz(π) ├─■─┤ √X ├─┤ Rz(π) ├─────■─┤ √X ├»
«          └────┘└───────┘   └────┘└───────┘   └────┘ └───────┘       └────┘»
«                     
«q_6 -> 18 ───────────
«          ┌─────────┐
«q_3 -> 29 ┤ Rz(π/2) ├
«          └─────────┘
«q_4 -> 30 ───────────
«          ┌─────────┐
«q_5 -> 31 ┤ Rz(π/2) ├
«          └─────────┘

In [28]:
paulis_opt[2].draw(idle_wires=False)

»
q_6 -> 18 ─────────────────────────────────────────────────────────────────────»
          ┌─────────┐┌────┐┌───────┐                                           »
q_1 -> 27 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─────────────────────────────────────────»
          └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐              »
q_2 -> 28 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────»
                                       └─────────┘└────┘└───────┘ │ ┌─────────┐»
q_3 -> 29 ────────────────────────────────────────────────────────■─┤ Rz(π/2) ├»
                                                                    └─────────┘»
q_4 -> 30 ─────────────────────────────────────────────────────────────────────»
                                                                               »
q_5 -> 31 ─────────────────────────────────────────────────────────────────────»
                                                                               »
«                                                                          »
«q_6 -> 18 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_1 -> 27 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_2 -> 28 ────────────────────────────────────────────────────────────────»
«          ┌────┐┌───────┐                                                 »
«q_3 -> 29 ┤ √X ├┤ Rz(π) ├─■───────────────────────────────────────────────»
«          └────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐                    »
«q_4 -> 30 ────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■──────────────────»
«                            └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐»
«q_5 -> 31 ─────────────────────────────────────────────■─┤ Rz(π/2) ├┤ √X ├»
«                                                         └─────────┘└────┘»
«                                                                            »
«q_6 -> 18 ──────────■───────────────────────────────────────────────────────»
«                    │                                                       »
«q_1 -> 27 ──────────┼───────────────────────────────────────────────────────»
«                    │                                                       »
«q_2 -> 28 ──────────┼─────────────────────────────────────────────────────■─»
«                    │                                     ┌────┐┌───────┐ │ »
«q_3 -> 29 ──────────┼───────────────────────────────────■─┤ √X ├┤ Rz(π) ├─■─»
«                    │                   ┌────┐┌───────┐ │ └────┘└───────┘   »
«q_4 -> 30 ──────────┼─────────────────■─┤ √X ├┤ Rz(π) ├─■───────────────────»
«          ┌───────┐ │ ┌────┐┌───────┐ │ └────┘└───────┘                     »
«q_5 -> 31 ┤ Rz(π) ├─■─┤ √X ├┤ Rz(π) ├─■─────────────────────────────────────»
«          └───────┘   └────┘└───────┘                                       »
«                                                                          »
«q_6 -> 18 ────────────────────────────────────────────────────────■───────»
«                              ┌────┐┌─────────┐                   │       »
«q_1 -> 27 ──────────────────■─┤ √X ├┤ Rz(π/2) ├───────────────────┼───────»
«          ┌────┐┌─────────┐ │ └────┘└─────────┘                   │       »
«q_2 -> 28 ┤ √X ├┤ Rz(π/2) ├─■─────────────────────────────────────┼───────»
«          ├────┤└┬───────┬┘                                       │       »
«q_3 -> 29 ┤ √X ├─┤ Rz(π) ├──■─────────────────────────────────────┼───────»
«          └────┘ └───────┘  │ ┌────┐ ┌───────┐                    │       »
«q_4 -> 30 ──────────────────■─┤ √X ├─┤ Rz(π) ├──■─────────────────┼───────»
«                              └────┘ └───────┘  │ ┌────┐┌───────┐ │ ┌────┐»
«q_5 -> 31 ──────────────────────────────────────■─┤ √X ├┤ Rz(π) ├─■─┤ √X ├»
«                                                  └────┘└───────┘   └────

In [29]:
def ApplyControlledPauliGate(ihd, qc_):
    for inst in paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)

In [30]:
# get a duration
from qiskit.transpiler import InstructionDurations
dur = InstructionDurations.from_backend(backend)
#print(dur)
t_cx_m1 = dur.get('cz',(initial_layout[indx_qm],initial_layout[index_attached_to_qm]))
#t_cx_m2 = dur.get('cz',(initial_layout[index_attached_to_qm-2],initial_layout[index_attached_to_qm-3]))
print(t_cx_m1*dur.dt)
#print(t_cx_m2*dur.dt)

6.8e-08


In [31]:
# find the position of CZ[qm,qr]
for ihd in range(nhd):
    k = 0
    for inst in paulis_opt[ihd].data:
        if (inst.name=='cz'):
            print(k,inst.qubits)
        k += 1 
    print('//')

3 (Qubit(QuantumRegister(133, 'q'), 31), Qubit(QuantumRegister(133, 'q'), 32))
7 (Qubit(QuantumRegister(133, 'q'), 18), Qubit(QuantumRegister(133, 'q'), 31))
10 (Qubit(QuantumRegister(133, 'q'), 31), Qubit(QuantumRegister(133, 'q'), 32))
//
3 (Qubit(QuantumRegister(133, 'q'), 30), Qubit(QuantumRegister(133, 'q'), 29))
7 (Qubit(QuantumRegister(133, 'q'), 31), Qubit(QuantumRegister(133, 'q'), 30))
11 (Qubit(QuantumRegister(133, 'q'), 18), Qubit(QuantumRegister(133, 'q'), 31))
14 (Qubit(QuantumRegister(133, 'q'), 31), Qubit(QuantumRegister(133, 'q'), 30))
17 (Qubit(QuantumRegister(133, 'q'), 30), Qubit(QuantumRegister(133, 'q'), 29))
20 (Qubit(QuantumRegister(133, 'q'), 18), Qubit(QuantumRegister(133, 'q'), 31))
//
3 (Qubit(QuantumRegister(133, 'q'), 28), Qubit(QuantumRegister(133, 'q'), 27))
7 (Qubit(QuantumRegister(133, 'q'), 29), Qubit(QuantumRegister(133, 'q'), 28))
11 (Qubit(QuantumRegister(133, 'q'), 30), Qubit(QuantumRegister(133, 'q'), 29))
15 (Qubit(QuantumRegister(133, 'q'), 31)

In [32]:
indx_cz = 7

# empty cxx circuit
empty_paulis_opt = []
circuit = QuantumCircuit(qr_circuit)
circuit = pm3.run(circuit)
ii = 0
for inst in paulis_opt[0].data:
    #print(inst)
    if ii==indx_cz:
        print(inst)
        circuit.barrier(initial_layout[indx_qm])
        circuit.barrier(initial_layout[index_attached_to_qm])
        circuit.delay(t_cx_m1,initial_layout[index_attached_to_qm])
        circuit.delay(t_cx_m1,initial_layout[indx_qm])
        circuit.barrier(initial_layout[indx_qm])
        circuit.barrier(initial_layout[index_attached_to_qm])
        #circuit.barrier()
    else:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ii += 1
empty_paulis_opt.append(circuit)

# n_site>=8, second cxx circuit
if (n_site>=8) and (n_dimer%2==0):
    circuit = QuantumCircuit(qr_circuit)
    circuit = pm3.run(circuit)
    indx_cz = [11, 20]
    ii = 0
    for inst in paulis_opt[1].data:
        #print(inst)
        if ii==indx_cz[0]:
            print(inst)
            circuit.cz(initial_layout[indx_qm],initial_layout[index_attached_to_qm])
            circuit.barrier(initial_layout[indx_qm])
            circuit.barrier(initial_layout[index_attached_to_qm])
            circuit.cz(initial_layout[indx_qm],initial_layout[index_attached_to_qm])
            #circuit.barrier()
        elif ii==indx_cz[1]:
            pass
        else:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        ii += 1
    empty_paulis_opt.append(circuit)

# n_site>=10, second cxx circuit
if (n_site>=10) and (n_dimer%2==1):
    circuit = QuantumCircuit(qr_circuit)
    circuit = pm3.run(circuit)
    indx_cz = [15, 32]
    ii = 0
    for inst in paulis_opt[1].data:
        #print(inst)
        if ii==indx_cz[0]:
            print(inst)
            circuit.cz(initial_layout[indx_qm],initial_layout[index_attached_to_qm])
            circuit.barrier(initial_layout[indx_qm])
            circuit.barrier(initial_layout[index_attached_to_qm])
            circuit.cz(initial_layout[indx_qm],initial_layout[index_attached_to_qm])
            #circuit.barrier()
        elif ii==indx_cz[1]:
            pass
        else:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        ii += 1
    empty_paulis_opt.append(circuit)

# n_site>=12, third cxx circuit
if (n_site>=12) and (n_dimer%2==0):
    circuit = QuantumCircuit(qr_circuit)
    circuit = pm3.run(circuit)
    indx_cz = [19, 42]
    ii = 0
    for inst in paulis_opt[2].data:
        #print(inst)
        if ii==indx_cz[0]:
            print(inst)
            circuit.cz(initial_layout[indx_qm],initial_layout[index_attached_to_qm])
            circuit.barrier(initial_layout[indx_qm])
            circuit.barrier(initial_layout[index_attached_to_qm])
            circuit.cz(initial_layout[indx_qm],initial_layout[index_attached_to_qm])
            #circuit.barrier()
        elif ii==indx_cz[1]:
            pass
        else:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        ii += 1
    empty_paulis_opt.append(circuit)



def ApplyEmptyPauliGate(ihd, qc_):
    for inst in empty_paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)

CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 18), Qubit(QuantumRegister(133, 'q'), 31)), clbits=())
CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 18), Qubit(QuantumRegister(133, 'q'), 31)), clbits=())
CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 18), Qubit(QuantumRegister(133, 'q'), 31)), clbits=())


In [33]:
observable_device = observable.apply_layout(paulis_opt[0].layout)
n_qubit_device = backend.num_qubits
pauli = observable_device.paulis[0]
for k in range(n_qubit_device):
    p = pauli[k]
    if (str(p)=='Z'):
        print(k,p)

18 Z


In [34]:
# for aer
# def ApplyControlledPauliGate(ihd, qc_):
#     for inst in paulis_opt[ihd].data:
#         qc_.append(inst.operation, inst.qubits, inst.clbits)
# empty_paulis_opt = []
# for ihd in range(nhd):
#     circuit = QuantumCircuit(qr_circuit)
#     empty_paulis_opt.append(circuit)
# def ApplyEmptyPauliGate(ihd, qc_):
#     for inst in empty_paulis_opt[ihd].data:
#         qc_.append(inst.operation, inst.qubits, inst.clbits)

In [35]:
empty_paulis_opt[0].draw(idle_wires=False)

┌─────────┐┌────┐┌───────┐ ░ »
q_5 -> 31 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─░─»
          ┌─────────┐┌────┐┌───────┐ │ └─────────┘└────┘└───────┘ ░ »
q_7 -> 32 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■──────────────────────────────»
          └─────────┘└────┘└───────┘                                »
«          ┌───────────────┐ ░ ┌────┐┌─────────┐                    
«q_5 -> 31 ┤ Delay(17[dt]) ├─░─┤ √X ├┤ Rz(π/2) ├─■──────────────────
«          └───────────────┘ ░ └────┘└─────────┘ │ ┌────┐┌─────────┐
«q_7 -> 32 ──────────────────────────────────────■─┤ √X ├┤ Rz(π/2) ├
«                                                  └────┘└─────────┘

In [36]:
empty_paulis_opt[1].draw(idle_wires=False)


»
q_6 -> 18 ─────────────────────────────────────────────────────────────────────»
          ┌─────────┐┌────┐┌───────┐                                           »
q_3 -> 29 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─────────────────────────────────────────»
          └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐              »
q_4 -> 30 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────»
                                       └─────────┘└────┘└───────┘ │ ┌─────────┐»
q_5 -> 31 ────────────────────────────────────────────────────────■─┤ Rz(π/2) ├»
                                                                    └─────────┘»
«                             ░                                             »
«q_6 -> 18 ────────────────■──░──■──────────────────────────────────────────»
«                          │  ░  │                                          »
«q_3 -> 29 ────────────────┼─────┼──────────────────────────────────────■───»
«                          │     │                   ┌────┐┌─────────┐  │   »
«q_4 -> 30 ────────────────┼─────┼─────────────────■─┤ √X ├┤ Rz(π/2) ├──■───»
«          ┌────┐┌───────┐ │  ░  │ ┌────┐┌───────┐ │ ├────┤└┬───────┬┘┌────┐»
«q_5 -> 31 ┤ √X ├┤ Rz(π) ├─■──░──■─┤ √X ├┤ Rz(π) ├─■─┤ √X ├─┤ Rz(π) ├─┤ √X ├»
«          └────┘└───────┘    ░    └────┘└───────┘   └────┘ └───────┘ └────┘»
«                                
«q_6 -> 18 ──────────────────────
«             ┌────┐  ┌─────────┐
«q_3 -> 29 ───┤ √X ├──┤ Rz(π/2) ├
«             └────┘  └─────────┘
«q_4 -> 30 ──────────────────────
«          ┌─────────┐           
«q_5 -> 31 ┤ Rz(π/2) ├───────────
«          └─────────┘

In [37]:
empty_paulis_opt[2].draw(idle_wires=False)

»
q_6 -> 18 ─────────────────────────────────────────────────────────────────────»
          ┌─────────┐┌────┐┌───────┐                                           »
q_1 -> 27 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─────────────────────────────────────────»
          └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐              »
q_2 -> 28 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────»
                                       └─────────┘└────┘└───────┘ │ ┌─────────┐»
q_3 -> 29 ────────────────────────────────────────────────────────■─┤ Rz(π/2) ├»
                                                                    └─────────┘»
q_4 -> 30 ─────────────────────────────────────────────────────────────────────»
                                                                               »
q_5 -> 31 ─────────────────────────────────────────────────────────────────────»
                                                                               »
«                                                                          »
«q_6 -> 18 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_1 -> 27 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_2 -> 28 ────────────────────────────────────────────────────────────────»
«          ┌────┐┌───────┐                                                 »
«q_3 -> 29 ┤ √X ├┤ Rz(π) ├─■───────────────────────────────────────────────»
«          └────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐                    »
«q_4 -> 30 ────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■──────────────────»
«                            └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐»
«q_5 -> 31 ─────────────────────────────────────────────■─┤ Rz(π/2) ├┤ √X ├»
«                                                         └─────────┘└────┘»
«                       ░                                              »
«q_6 -> 18 ──────────■──░──■───────────────────────────────────────────»
«                    │  ░  │                                           »
«q_1 -> 27 ──────────┼─────┼───────────────────────────────────────────»
«                    │     │                                           »
«q_2 -> 28 ──────────┼─────┼───────────────────────────────────────────»
«                    │     │                                     ┌────┐»
«q_3 -> 29 ──────────┼─────┼───────────────────────────────────■─┤ √X ├»
«                    │     │                   ┌────┐┌───────┐ │ └────┘»
«q_4 -> 30 ──────────┼─────┼─────────────────■─┤ √X ├┤ Rz(π) ├─■───────»
«          ┌───────┐ │  ░  │ ┌────┐┌───────┐ │ └────┘└───────┘         »
«q_5 -> 31 ┤ Rz(π) ├─■──░──■─┤ √X ├┤ Rz(π) ├─■─────────────────────────»
«          └───────┘    ░    └────┘└───────┘                           »
«                                                                             »
«q_6 -> 18 ───────────────────────────────────────────────────────────────────»
«                                          ┌────┐┌─────────┐                  »
«q_1 -> 27 ──────────────────────────────■─┤ √X ├┤ Rz(π/2) ├──────────────────»
«                      ┌────┐┌─────────┐ │ └────┘└─────────┘                  »
«q_2 -> 28 ──────────■─┤ √X ├┤ Rz(π/2) ├─■────────────────────────────────────»
«          ┌───────┐ │ ├────┤└┬───────┬┘                                      »
«q_3 -> 29 ┤ Rz(π) ├─■─┤ √X ├─┤ Rz(π) ├──■────────────────────────────────────»
«          └───────┘   └────┘ └───────┘  │ ┌────┐ ┌───────┐                   »
«q_4 -> 30 ──────────────────────────────■─┤ √X ├─┤ Rz(π) ├──■────────────────»
«                                          └────┘ └───────┘  │ ┌────┐┌───────┐»
«q_5 -> 31 ──────────────────────────────────────────────────■─┤ √X ├┤ Rz(π) ├»
«                                                              └────┘└───────┘»
«                                   

In [38]:
#circuit = QuantumCircuit(qr_circuit)
#ApplyControlledPauliGate(0,circuit)
#print(circuit)
#ApplyControlledPauliGate(1,circuit)
#print(circuit)

In [17]:
eigen_energies_ref = np.zeros((n_hamiltonians),dtype=float)

for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    eigen_energies_ref[i_inter] = - J/4 * Delta * (n_dimer)
    # inter-dimer energies
    eigen_energies_ref[i_inter] += - J/4 * Delta * t_inter* (n_dimer-1) # open boundary condition

In [18]:
print(eigen_energies_ref)

[1.5  2.75]


In [19]:
e_dimer = -0.25 * J * (2.0-Delta)
print (e_dimer)

-0.75


In [20]:
norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)


eigen_energies_qzmc[0]  = e_dimer*n_dimer
print(eigen_energies_qzmc[0])

-4.5


In [21]:
pauli_identity = 'I'*n_qubit

In [44]:
#def fake_run_estimator (max_circuits_, estimator_, pubs_):
#    len_circuits_ = len(pubs_)
#    num_jobs_     = len_circuits_//max_circuits_
#    remainder_    = len_circuits_%max_circuits_
#    if (remainder_>0):
#        num_jobs_ += 1
#    i_start_ = 0
#    list_out = []
#    for _ in range(num_jobs_):
#        i_end_ = min(i_start_ + max_circuits_,len_circuits_)
#        job_ = estimator_.run(pubs_[i_start_:i_end_])
#        job_result = job_.result()
#        len_result = len(job_result)
#        for i in range(len_result):
#            list_out.append(job_result[i].data.evs)
#        i_start_ += max_circuits_
#    return list_out

In [22]:
def run_estimator (max_circuits_, estimator_, pubs_):
    len_circuits_ = len(pubs_)
    num_jobs_     = len_circuits_//max_circuits_
    remainder_    = len_circuits_%max_circuits_
    if (remainder_>0):
        num_jobs_ += 1
    i_start_ = 0
    job_ids_ = []
    for _ in range(num_jobs_):
        i_end_ = min(i_start_ + max_circuits_,len_circuits_)
        job_ = estimator_.run(pubs_[i_start_:i_end_])
        job_ids_.append(job_.job_id())
        i_start_ += max_circuits_
    return job_ids_

In [23]:
import time as time_lib

def moniter_jobs (service_, job_ids_):
    num_jobs_       = len(job_ids_)
    done_           = False
    problem_        = False
    while not done_:
        done_ = True
        i_start_ = 0
        for ind_job_ in range(num_jobs_):
            job_id_ = job_ids_[ind_job_]
            job_ = service_.job(job_id_)
            if job_.status() == 'DONE':
                continue
            else:
                done_ = False
                # exit if there is a problem
                if job_.status() in ['ERROR', 'CANCELLED']:
                    s_ = ''
                    s_ += '## There was a problem in submitting job_ids[{ind_job}], '.format(ind_job=ind_job_)
                    s_ +='\n'
                    s_ += '  {}'.format(job_id_)
                    s_ +='\n'
                    print(s_)
                    problem_ = True
                if (problem_):
                    break
        if (problem_):
            break
        time_lib.sleep(30)
    return [done_, problem_]

def accumulate_job_results (service_, job_ids_):
    list_out = []
    for job_id_ in job_ids_:
        job_ = service_.job(job_id_)
        while job_.status() != 'DONE':
            time_lib.sleep(30) 
        job_result = job_.result()
        len_result = len(job_result)
        for i in range(len_result):
            list_out.append(job_result[i].data.evs)
    return list_out


In [47]:
# for aer
#run_circuits_file = 'circuits_opt.qpy'
#if (core==0):
#    code_run_estimator = '''from qiskit_aer.primitives import EstimatorV2 as Estimator
#from qiskit import qpy
#import sys, pickle
#from qiskit.quantum_info import SparsePauliOp
#
#
#pickled_input_data = sys.stdin.buffer.read()
#input_data = pickle.loads(pickled_input_data)
#n_qubit ='''+ str(n_qubit) + '''
#observable = SparsePauliOp.from_sparse_list([("Z", ['''+str(indx_qm)+'''], 1)], num_qubits='''+str(n_qubit_circuit)+''')
#backend_options = '''+ str(backend_options) +'''
#estimator = Estimator(options = {"default_precision":0.00,
#                                     'backend_options':backend_options})
#run_circuits_file = \'''' + run_circuits_file + '''\'
#with open(run_circuits_file, 'rb') as file_:
#    circuits = qpy.load(file_)
#pubs = []
#len_input = len(input_data)
#for ind in range(len_input):
#    i0     = input_data[ind][0]
#    if (len(input_data[ind])>1):
#        params = input_data[ind][1]
#        pubs.append((circuits[i0],observable,params))
#    else:
#        pubs.append((circuits[i0],observable))
#job = estimator.run(pubs)
#result = job.result()
#len_result = len(result)
#list_out = []
#for i in range(len_result):
#    list_out.append(result[i].data.evs)
#sys.stdout.buffer.write(pickle.dumps(list_out))
#    '''
#    
#    with open('run_estimator.py', 'w') as file_:
#        file_.write(code_run_estimator)

In [24]:
from datetime import datetime
from qiskit.circuit import ParameterVector

In [25]:
job_ids_save = [None for _ in range(n_hamiltonians)]
num_jobs_save = [None for _ in range(n_hamiltonians)]

In [26]:
# read 10 qubit QZMC values
ten_qubit_dir = '../../10/ibm_torino'
n_alpha_10 = 1
e_qzmc_10_qubit = 0.0
with open(ten_qubit_dir + '/qzmc.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        if (alpha==n_alpha_10):
            e_qzmc_10_qubit = float(ls[-1])
        alpha += 1
print(e_qzmc_10_qubit)

-4.299440799638228


In [27]:
eps = e_dimer+e_qzmc_10_qubit
## first-order perturbation correction = 0
# predictor from 10 and 2 qubit values
print(eps)

-5.049440799638228


In [52]:
# first run, dE1
alpha = 1

start_time = datetime.now()
circuits = []
times = ParameterVector('t', 2*alpha-1)

# implement quantum circuits by parts
circuit_segment = [None for _ in range(2)] 
circuit = QuantumCircuit(qr_circuit)
circuit.h(qm)
for inst in init_circuit.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)
i_time  = 0
phase   = 0.0
# \mathcal{P}_{\alpha-1}
evolution_times = []
alphas = []
n_trotters = []
for alpha_ in range(1,alpha):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
if (len(evolution_times)>0):
    phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[0]/n_trotters[0]/2
ApplyConsecutiveTrotterEvolution_from_dimer_solution(evolution_times,alphas,n_trotters,circuit)
circuit_segment[0] = pm3.run(circuit)
# circuit_segment[0] = circuit

circuit = QuantumCircuit(qr_circuit)

evolution_times = []
alphas = []
n_trotters = []

# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
#print(evolution_times,alphas,n_trotters)
phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

# \mathcal{P}^{\dagger}_{\alpha-1}
for alpha_ in reversed(range(1,alpha)):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
ApplyConsecutiveTrotterEvolution_to_dimer_solution(evolution_times,alphas,n_trotters,circuit)
phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[-1]/n_trotters[-1]/2
#print(circuit)

for inst in init_circuit_inv.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)

circuit.rz(phase,qm)
circuit.h(qm)

circuit_segment[1] = pm3.run(circuit)
#circuit_segment[1] = circuit

nhd = len(hamiltonian_diffs_reduced_list[alpha-1])

for ihd in range(nhd):
    # norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyEmptyPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    #circuits.append(circuit) # aer
    circuit_opt = pm1.run(circuit)
    circuits.append(circuit_opt)
    del circuit_opt

for ihd in range(nhd):
    # dE*norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyControlledPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    #circuits.append(circuit) # aer
    circuit_opt = pm1.run(circuit)
    circuits.append(circuit_opt)
    del circuit_opt


# 
pubs = []

n_pubs = (2*nhd) * nmc  # nhd for norm, nhd for dE
n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1
if (core==0 and alpha==1):
    print('# of different quantum circuits to run = ', n_pubs)

i_start         = sum(n_pubs_for_[:core])
i_end           = i_start + n_pubs_for_[core]
ind_pub         = 0
indx_circuit    = 0
## norm
i_obs = 0
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
        ind_pub += 1
    indx_circuit += 1
# dE*norm
i_obs = 1
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
        ind_pub += 1
    indx_circuit += 1

# if (core==0):
#     with open ('circuits.qpy', 'wb') as file_:
#         qpy.dump(circuits,file_)
#     del circuits[:]
#     subprocess.run(['python3', 'run_pass_manager.py'])
# #comm.Barrier()

# estimator_inputs = []
# 
# n_pubs = 0
# # dE1
# n_pubs += (2*nhd) * nmc  # nhd for norm, nhd for dE
# 
# n_pubs_for_ = [0 for _ in range(cores)]
# remainder         = n_pubs%cores
# for i_core in range(cores):
#     n_pubs_for_[i_core] = n_pubs//cores
#     if (i_core<remainder):
#         n_pubs_for_[i_core] += 1
# if (core==0 and alpha==1):
#     print('# of different quantum circuits to run = ', n_pubs)
# 
# i_start         = sum(n_pubs_for_[:core])
# i_end           = i_start + n_pubs_for_[core]
# ind_pub         = 0
# indx_circuit    = 0
# ## norm
# i_obs = 0
# for ihd in range(nhd):
#     for imc in range(nmc):
#         my_turn = ind_pub>=i_start and ind_pub<i_end
#         if (my_turn):
#             estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
#         ind_pub += 1
#     indx_circuit += 1
# ## dE*norm
# i_obs = 1
# for ihd in range(nhd):
#     for imc in range(nmc):
#         my_turn = ind_pub>=i_start and ind_pub<i_end
#         if (my_turn):
#             estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
#         ind_pub += 1
#     indx_circuit += 1



# # for a fake backend
# result = fake_run_estimator(max_circuits, estimator, pubs)
job_ids = run_estimator(max_circuits, estimator, pubs)
job_ids_save[alpha] = job_ids
num_jobs_save[alpha] = len(job_ids)
with open('XXZ'+str(n_site)+'.job_ids','w') as file_:
    for alpha_ in range(1, n_hamiltonians):
        s = ''
        for job_id in job_ids_save[alpha_]:
            s += '  {}'.format(job_id)
        s += '\n'
        file_.write(s)


#gc.collect()
#serialized_estimator_inputs = pickle.dumps(estimator_inputs)
#estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
#               stdout=subprocess.PIPE,stderr=subprocess.PIPE)



# of different quantum circuits to run =  1800


In [28]:
with open('XXZ'+str(n_site)+'.job_ids','r') as file_:
    lines = file_.readlines()
    ind_line = 0
    for alpha in range(1,n_hamiltonians):
        line = lines[ind_line]
        job_ids = (line.split())
        job_ids_save[alpha]=job_ids
        num_jobs_save[alpha] = len(job_ids)
        print(job_ids_save[alpha])
        ind_line += 1

['cx1e2d9px23g008y6z80', 'cx1e2fsrkac0008zyb50', 'cx1e2j2ztp30008et0n0', 'cx1e2mapjw300082v6sg', 'cx1e2pjpx23g008y715g', 'cx1e2rvpjw300082v7mg']


In [29]:
#[done, problem] = moniter_jobs (service, job_ids_save[alpha])
start_time = datetime.now()
alpha = 1
nhd = len(hamiltonian_diffs_reduced_list[alpha-1])
result = accumulate_job_results(service, job_ids_save[alpha])
print('accumulation done')
#try:
#    result = pickle.loads(estimator_run.stdout)
#except EOFError: # empty list
#    result = []
gc.collect()
#
#for i in range(n_pubs_for_[core]):
#    #print(result[i])
#    computed_value = result[i]
#
#    # # shot errors
#    #p_up = (computed_value + 1.0)/2.0
#    #sample = np.random.binomial(n_shot,p_up)
#    #shot_error = 2*(sample/(n_shot) - p_up)
#    #computed_value += shot_error
#
#    result_values_core[i] = computed_value
##print(core,result_values_core)

n_pubs = (2*nhd) * nmc  # nhd for norm, nhd for dE
n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1

result_values_core = [0.0 for _ in range(n_pubs_for_[core])]

for i in range(n_pubs_for_[core]):
    result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
del result


### bcast
#comm.Barrier()
result_values = []
for i_core in range(cores):
    if (n_pubs_for_[i_core]==0):
        continue
    result_values_temp = result_values_core
    result_values += result_values_temp
    #comm.Barrier()
print(result_values)

norms_1    = np.zeros((nhd),dtype=float)
pauli_norms_1  = np.zeros((nhd),dtype=float)
i_meas = 0
# norms
for ihd in range(nhd):
    for imc in range(nmc):
        norms_1[ihd] += result_values[i_meas]
        i_meas += 1
# paulis
for ihd in range(nhd):
    for imc in range(nmc):
        pauli_norms_1[ihd] += result_values[i_meas]
        i_meas += 1
norms_1 /= nmc
pauli_norms_1 /= nmc

print(norms_1)
print(pauli_norms_1)

# dE1
dE1 = 0.0
for ihd in range(nhd):
    pauli_exp = pauli_norms_1[ihd]/norms_1[ihd]
    coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
    dE1 += coeff * pauli_exp
dE1 = dE1.real

eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
norms_qzmc[alpha] = np.sum(norms_1)/(nhd)
del result_values[:]
del result_values_core[:]
gc.collect()

end_time = datetime.now()
elapsed = end_time-start_time
elapsed = elapsed.total_seconds()
if (core==0):
    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
    memory_usage(st)
    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
    print(st)

accumulation done
[array(0.24802372), array(0.35079051), array(0.15513834), array(0.0513834), array(0.15711462), array(0.22134387), array(0.07806324), array(0.36166008), array(-0.02667984), array(0.1215415), array(0.07608696), array(0.31324111), array(0.13932806), array(0.40217391), array(0.20454545), array(0.10375494), array(0.03853755), array(0.23814229), array(0.08992095), array(0.38735178), array(0.38142292), array(0.21343874), array(0.1541502), array(0.07509881), array(0.04249012), array(0.33102767), array(0.08695652), array(0.20256917), array(0.02766798), array(0.10968379), array(0.2687747), array(0.22134387), array(0.24703557), array(0.08893281), array(0.08003953), array(0.08893281), array(0.11660079), array(0.03656126), array(0.35671937), array(0.06521739), array(0.0187747), array(0.10375494), array(0.25494071), array(0.05434783), array(0.10177866), array(-0.07411067), array(0.08102767), array(0.24407115), array(0.28063241), array(0.02766798), array(0.35079051), array(-0.000988

In [30]:
# save to file
with open('qzmc.norm.E.save','w') as file_:
    s = '# t_inter, norms, E'
    s += '\n'
    file_.write(s)
    for alpha in range(n_inter):
        t_inter = t_inters[alpha]
        s = '{:}'.format(t_inter)
        s += '  {:.16e}'.format(norms_qzmc[alpha])
        s += '  {:.16e}'.format(eigen_energies_qzmc[alpha])
        print(s)
        s += '\n'
        file_.write(s)

0.0  1.0000000000000000e+00  -4.5000000000000000e+00
1.0  1.4159962896317305e-01  -5.0915013185539602e+00


In [101]:
# first run, dE2
alpha = 1

start_time = datetime.now()
circuits = []
times = ParameterVector('t', 2*alpha)

# implement quantum circuits by parts
circuit_segment = [None for _ in range(2)] 
circuit = QuantumCircuit(qr_circuit)
circuit.h(qm)
for inst in init_circuit.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)
i_time  = 0
phase   = 0.0
# \mathcal{P}_{\alpha-1}
evolution_times = []
alphas = []
n_trotters = []
for alpha_ in range(1,alpha):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
phase += (eigen_energies_qzmc[alpha]-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

if (len(evolution_times)>0):
    phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[0]/n_trotters[0]/2
ApplyConsecutiveTrotterEvolution_from_dimer_solution(evolution_times,alphas,n_trotters,circuit)
circuit_segment[0] = circuit
#circuit_segment[0] = pm3.run(circuit)

circuit = QuantumCircuit(qr_circuit)

evolution_times = []
alphas = []
n_trotters = []

# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
phase += (eigen_energies_qzmc[alpha]-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

# \mathcal{P}^{\dagger}_{\alpha-1}
for alpha_ in reversed(range(1,alpha)):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
ApplyConsecutiveTrotterEvolution_to_dimer_solution(evolution_times,alphas,n_trotters,circuit)
phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[-1]/n_trotters[-1]/2
#print(circuit)

for inst in init_circuit_inv.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)

circuit.rz(phase,qm)
circuit.h(qm)

#circuit_segment[indx][1] = pm3.run(circuit)
circuit_segment[1] = circuit

nhd = len(hamiltonian_diffs_reduced_list[alpha])

for ihd in range(nhd):
    # norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyEmptyPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer

for ihd in range(nhd):
    # dE*norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyControlledPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt


# 
# pubs = []
# 
# n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
# n_pubs_for_ = [0 for _ in range(cores)]
# remainder         = n_pubs%cores
# for i_core in range(cores):
#     n_pubs_for_[i_core] = n_pubs//cores
#     if (i_core<remainder):
#         n_pubs_for_[i_core] += 1
# if (core==0 and alpha==1):
#     print('# of different quantum circuits to run = ', n_pubs)
# 
# i_start         = sum(n_pubs_for_[:core])
# i_end           = i_start + n_pubs_for_[core]
# ind_pub         = 0
# indx_circuit    = 0
# ## dE1
# for ihd in range(nhd):
#     # norm, indx = 0, 1
#     i_obs = 0
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1
#     # dE*norm, indx = 0, 1
#     i_obs = 1
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1

if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    del circuits[:]
    subprocess.run(['python3', 'run_pass_manager.py'])
#comm.Barrier()

estimator_inputs = []

n_pubs = 0
# dE1
n_pubs += (2*nhd) * nmc  # nhd for norm, nhd for dE

n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1
if (core==0 and alpha==1):
    print('# of different quantum circuits to run = ', n_pubs)

i_start         = sum(n_pubs_for_[:core])
i_end           = i_start + n_pubs_for_[core]
ind_pub         = 0
indx_circuit    = 0
## norm
i_obs = 0
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc])]
        ind_pub += 1
    indx_circuit += 1
## dE2*norm
i_obs = 2
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc])]
        ind_pub += 1
    indx_circuit += 1


result_values_core = [0.0 for _ in range(n_pubs_for_[core])]


#job_ids = run_estimator(max_circuits, estimator, pubs)
#result = fake_run_estimator(max_circuits, estimator, pubs)
#job_ids_save[alpha] = job_ids
#num_jobs_save[alpha] = len(job_ids)
#with open('XXZ4.job_ids','w') as file_:
#    for alpha_ in range(1, n_hamiltonians):
#        s = ''
#        for job_id in job_ids_save[alpha_]:
#            s += '  {}'.format(job_id)
#        s += '\n'
#        file_.write(s)
#[done, problem] = moniter_jobs (service, job_ids_save[alpha])
#result = accumulate_job_results(service, job_ids_save[alpha])

#gc.collect()
#
#for i in range(n_pubs_for_[core]):
#    result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
#del result

gc.collect()
serialized_estimator_inputs = pickle.dumps(estimator_inputs)
estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
               stdout=subprocess.PIPE,stderr=subprocess.PIPE)



IndexError: list index out of range

In [ ]:
try:
    result = pickle.loads(estimator_run.stdout)
except EOFError: # empty list
    result = []
gc.collect()

for i in range(n_pubs_for_[core]):
    #print(result[i])
    computed_value = result[i]

    # # shot errors
    #p_up = (computed_value + 1.0)/2.0
    #sample = np.random.binomial(n_shot,p_up)
    #shot_error = 2*(sample/(n_shot) - p_up)
    #computed_value += shot_error

    result_values_core[i] = computed_value
#print(core,result_values_core)
del result


### bcast
#comm.Barrier()
result_values = []
for i_core in range(cores):
    if (n_pubs_for_[i_core]==0):
        continue
    result_values_temp = result_values_core
    result_values += result_values_temp
    #comm.Barrier()
#print(result_values)

norms_2    = np.zeros((nhd),dtype=float)
pauli_norms_2  = np.zeros((nhd),dtype=float)
i_meas = 0
# norms
for ihd in range(nhd):
    for imc in range(nmc):
        norms_2[ihd] += result_values[i_meas]
        i_meas += 1
# paulis
for ihd in range(nhd):
    for imc in range(nmc):
        pauli_norms_2[ihd] += result_values[i_meas]
        i_meas += 1
norms_2 /= nmc
pauli_norms_2 /= nmc

print(norms_2)
print(pauli_norms_2)

# dE2
dE2 = 0.0
for ihd in range(nhd):
    pauli_exp = pauli_norms_2[ihd]/norms_2[ihd]
    coeff = hamiltonian_diffs_reduced_list[alpha][ihd][1]
    dE2 += coeff * pauli_exp
dE2 = dE2.real

eps = eigen_energies_qzmc[alpha] + dE2
del times
del result_values[:]
del result_values_core[:]
gc.collect()

end_time = datetime.now()
elapsed = end_time-start_time
elapsed = elapsed.total_seconds()
if (core==0):
    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
    memory_usage(st)
    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
    if (alpha<n_hamiltonians-1):
        print('precision of the predictor for next', eps-eigen_energies_exact[-1,alpha+1])
    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
    print(st)

[0.58590867 0.58590867]
[0.1695538  0.16424088]
[# 50.0%, elapsed time = 14.327486 secs] memory usage:    0.25058 GiB
1 0.7458153963699342 -0.03689108840046629
precision of the predictor for next -0.003124601255234616
# 50.0%


In [51]:
# second run, dE1
alpha = 2

start_time = datetime.now()
circuits = []
times = ParameterVector('t', 2*alpha-1)

# implement quantum circuits by parts
circuit_segment = [None for _ in range(2)] 
circuit = QuantumCircuit(qr_circuit)
circuit.h(qm)
for inst in init_circuit.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)
i_time  = 0
phase   = 0.0
# \mathcal{P}_{\alpha-1}
evolution_times = []
alphas = []
n_trotters = []
for alpha_ in range(1,alpha):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
if (len(evolution_times)>0):
    phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[0]/n_trotters[0]/2
ApplyConsecutiveTrotterEvolution_from_dimer_solution(evolution_times,alphas,n_trotters,circuit)
circuit_segment[0] = circuit
#circuit_segment[0] = pm3.run(circuit)

circuit = QuantumCircuit(qr_circuit)

evolution_times = []
alphas = []
n_trotters = []

# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
#print(evolution_times,alphas,n_trotters)
phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

# \mathcal{P}^{\dagger}_{\alpha-1}
for alpha_ in reversed(range(1,alpha)):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
ApplyConsecutiveTrotterEvolution_to_dimer_solution(evolution_times,alphas,n_trotters,circuit)
phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[-1]/n_trotters[-1]/2
#print(circuit)

for inst in init_circuit_inv.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)

circuit.rz(phase,qm)
circuit.h(qm)

#circuit_segment[indx][1] = pm3.run(circuit)
circuit_segment[1] = circuit

nhd = len(hamiltonian_diffs_reduced_list[alpha-1])

for ihd in range(nhd):
    # norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyEmptyPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt

for ihd in range(nhd):
    # dE*norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyControlledPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt


# 
# pubs = []
# 
# n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
# n_pubs_for_ = [0 for _ in range(cores)]
# remainder         = n_pubs%cores
# for i_core in range(cores):
#     n_pubs_for_[i_core] = n_pubs//cores
#     if (i_core<remainder):
#         n_pubs_for_[i_core] += 1
# if (core==0 and alpha==1):
#     print('# of different quantum circuits to run = ', n_pubs)
# 
# i_start         = sum(n_pubs_for_[:core])
# i_end           = i_start + n_pubs_for_[core]
# ind_pub         = 0
# indx_circuit    = 0
# ## dE1
# for ihd in range(nhd):
#     # norm, indx = 0, 1
#     i_obs = 0
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1
#     # dE*norm, indx = 0, 1
#     i_obs = 1
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1

if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    del circuits[:]
    subprocess.run(['python3', 'run_pass_manager.py'])
#comm.Barrier()

estimator_inputs = []

n_pubs = 0
# dE1
n_pubs += (2*nhd) * nmc  # nhd for norm, nhd for dE

n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1
if (core==0 and alpha==1):
    print('# of different quantum circuits to run = ', n_pubs)

i_start         = sum(n_pubs_for_[:core])
i_end           = i_start + n_pubs_for_[core]
ind_pub         = 0
indx_circuit    = 0
## norm
i_obs = 0
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
        ind_pub += 1
    indx_circuit += 1
## dE*norm
i_obs = 1
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
        ind_pub += 1
    indx_circuit += 1


result_values_core = [0.0 for _ in range(n_pubs_for_[core])]


#job_ids = run_estimator(max_circuits, estimator, pubs)
#result = fake_run_estimator(max_circuits, estimator, pubs)
#job_ids_save[alpha] = job_ids
#num_jobs_save[alpha] = len(job_ids)
#with open('XXZ4.job_ids','w') as file_:
#    for alpha_ in range(1, n_hamiltonians):
#        s = ''
#        for job_id in job_ids_save[alpha_]:
#            s += '  {}'.format(job_id)
#        s += '\n'
#        file_.write(s)
#[done, problem] = moniter_jobs (service, job_ids_save[alpha])
#result = accumulate_job_results(service, job_ids_save[alpha])

#gc.collect()
#
#for i in range(n_pubs_for_[core]):
#    result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
#del result

gc.collect()
serialized_estimator_inputs = pickle.dumps(estimator_inputs)
estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
               stdout=subprocess.PIPE,stderr=subprocess.PIPE)



In [52]:
try:
    result = pickle.loads(estimator_run.stdout)
except EOFError: # empty list
    result = []
gc.collect()

for i in range(n_pubs_for_[core]):
    #print(result[i])
    computed_value = result[i]

    # # shot errors
    #p_up = (computed_value + 1.0)/2.0
    #sample = np.random.binomial(n_shot,p_up)
    #shot_error = 2*(sample/(n_shot) - p_up)
    #computed_value += shot_error

    result_values_core[i] = computed_value
#print(core,result_values_core)
del result


### bcast
#comm.Barrier()
result_values = []
for i_core in range(cores):
    if (n_pubs_for_[i_core]==0):
        continue
    result_values_temp = result_values_core
    result_values += result_values_temp
    #comm.Barrier()
#print(result_values)

norms_1    = np.zeros((nhd),dtype=float)
pauli_norms_1  = np.zeros((nhd),dtype=float)
i_meas = 0
# norms
for ihd in range(nhd):
    for imc in range(nmc):
        norms_1[ihd] += result_values[i_meas]
        i_meas += 1
# paulis
for ihd in range(nhd):
    for imc in range(nmc):
        pauli_norms_1[ihd] += result_values[i_meas]
        i_meas += 1
norms_1 /= nmc
pauli_norms_1 /= nmc

print(norms_1)
print(pauli_norms_1)

# dE1
dE1 = 0.0
for ihd in range(nhd):
    pauli_exp = pauli_norms_1[ihd]/norms_1[ihd]
    coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
    dE1 += coeff * pauli_exp
dE1 = dE1.real

eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
norms_qzmc[alpha] = np.sum(norms_1)/(nhd)
del times
del result_values[:]
del result_values_core[:]
gc.collect()

end_time = datetime.now()
elapsed = end_time-start_time
elapsed = elapsed.total_seconds()
if (core==0):
    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
    memory_usage(st)
    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
    print(st)

[0.50190114 0.50190114]
[0.15115243 0.14173032]
[# 100.0%, elapsed time = 18.064495 secs] memory usage:    0.25058 GiB
2 0.5019011365208734 -0.008315508755019074
# 100.0%
